# Parsed and Annotated Data

DS 5001 Text as Data


**Purpose**: Import the constitution collection, convert it to F2 tables, and annotate it to create an F3-level model.

This notebook follows the M04 pipeline pattern from class: configure paths, inspect structure, register source files in `LIB`, tokenize the collection into `CORPUS`, extract document features, and build/annotate `VOCAB`.


# Set Up

## Config

In [1]:
data_home = "."
output_dir = "output"
source_file_dir = f"{data_home}/data"
data_prefix = "constitutions"

CSV_DELIM = "|"


In [2]:
import math
from glob import glob
from pathlib import Path
import re

import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import RegexpTokenizer

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

for package in ["punkt", "punkt_tab", "averaged_perceptron_tagger_eng", "stopwords"]:
    nltk.download(package, quiet=True)


# Inspect

The constitution files vary in markup, so we define a common OHCO and a small set of heading patterns. These patterns convert country-specific labels such as `PART`, `TITLE`, `CHAPTER`, `ARTICLE`, and numbered provisions into a regular article-like structure.


In [3]:
# This structure matches all the texts well enough for corpus analysis.
OHCO = [
    "constitution_id",
    "part_num",
    "section_num",
    "unit_num",
    "para_num",
    "sent_num",
    "token_num",
]

# Common website artifact to remove from all files.
clip_pats = [
    r"^Share$"
]

roman = "[IVXLCDM]+"
heading_pat_list = [
    ("preamble",   r"^Preamble$"),
    ("part",       r"^(?:PART|Part)\b"),
    ("title",      r"^(?:TITLE|Title)\b"),
    ("chapter",    r"^(?:CHAPTER|Chapter)\b"),
    ("schedule",   r"^(?:SCHEDULE|Schedule)\b"),
    ("article",    rf"^(?:ARTICLE|Article)\s+(?:\d+[A-Z]?|{roman})\b"),
    ("section",    r"^(?:SECTION|Section)\s+[A-Za-z0-9]+\b"),
    ("numbered",   r"^\d+[A-Z]?\.\s+\S+"),
]
heading_pat_list


[('preamble', '^Preamble$'),
 ('part', '^(?:PART|Part)\\b'),
 ('title', '^(?:TITLE|Title)\\b'),
 ('chapter', '^(?:CHAPTER|Chapter)\\b'),
 ('schedule', '^(?:SCHEDULE|Schedule)\\b'),
 ('article', '^(?:ARTICLE|Article)\\s+(?:\\d+[A-Z]?|[IVXLCDM]+)\\b'),
 ('section', '^(?:SECTION|Section)\\s+[A-Za-z0-9]+\\b'),
 ('numbered', '^\\d+[A-Z]?\\.\\s+\\S+')]

# Register

We get each source file and add it to the `LIB` table. The `constitution_id` comes from the filename.


In [4]:
source_file_list = sorted(glob(f"{source_file_dir}/*.txt"))
source_file_list[:10], len(source_file_list)


(['./data/Afghanistan_2004.txt',
  './data/Albania_2008.txt',
  './data/Algeria_2008.txt',
  './data/Andorra_1993.txt',
  './data/Angola_2010.txt',
  './data/Antigua_and_Barbuda_1981.txt',
  './data/Argentina_1994.txt',
  './data/Armenia_2005.txt',
  './data/Australia_1985.txt',
  './data/Austria_2009.txt'],
 192)

In [5]:
def parse_filename(source_file_path):
    stem = Path(source_file_path).stem
    country, file_year = stem.rsplit("_", 1)
    return stem, country.replace("_", " "), int(file_year)


def parse_title_line(line):
    title_pat = re.compile(r"^(?P<title>.*?)(?:\s+(?P<original_year>\d{4}))?(?:\s+\(rev\.\s*(?P<revision_year>\d{4})\))?\s*$")
    match = title_pat.match(line.strip())
    if not match:
        return line.strip(), pd.NA, pd.NA
    original_year = match.group("original_year")
    revision_year = match.group("revision_year")
    return (
        match.group("title").strip(),
        int(original_year) if original_year else pd.NA,
        int(revision_year) if revision_year else pd.NA,
    )

book_data = []
for source_file_path in source_file_list:
    constitution_id, country, file_year = parse_filename(source_file_path)
    raw_title = Path(source_file_path).read_text(encoding="utf-8", errors="ignore").splitlines()[0]
    book_data.append((constitution_id, source_file_path, raw_title, country, file_year))

book_data[:5]


[('Afghanistan_2004',
  './data/Afghanistan_2004.txt',
  'Afghanistan 2004',
  'Afghanistan',
  2004),
 ('Albania_2008',
  './data/Albania_2008.txt',
  'Albania 1998 (rev. 2008)',
  'Albania',
  2008),
 ('Algeria_2008',
  './data/Algeria_2008.txt',
  'Algeria 1963 (rev. 2008)',
  'Algeria',
  2008),
 ('Andorra_1993', './data/Andorra_1993.txt', 'Andorra 1993', 'Andorra', 1993),
 ('Angola_2010', './data/Angola_2010.txt', 'Angola 2010', 'Angola', 2010)]

In [6]:
LIB = pd.DataFrame(
    book_data,
    columns=["constitution_id", "source_file_path", "raw_title", "country", "file_year"]
).set_index("constitution_id").sort_index()
LIB.head()


,source_file_path,raw_title,country,file_year
constitution_id,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan 2004,Afghanistan,2004
Albania_2008,./data/Albania_2008.txt,Albania 1998 (rev. 2008),Albania,2008
Algeria_2008,./data/Algeria_2008.txt,Algeria 1963 (rev. 2008),Algeria,2008
Andorra_1993,./data/Andorra_1993.txt,Andorra 1993,Andorra,1993
Angola_2010,./data/Angola_2010.txt,Angola 2010,Angola,2010


In [7]:
title_data = LIB.raw_title.apply(parse_title_line).apply(pd.Series)
title_data.columns = ["title", "original_year", "revision_year"]
LIB = LIB.join(title_data).drop("raw_title", axis=1)
LIB.head()


,source_file_path,country,file_year,title,original_year,revision_year
constitution_id,,,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan,2004,Afghanistan,2004.0,<NA>
Albania_2008,./data/Albania_2008.txt,Albania,2008,Albania,1998.0,2008
Algeria_2008,./data/Algeria_2008.txt,Algeria,2008,Algeria,1963.0,2008
Andorra_1993,./data/Andorra_1993.txt,Andorra,1993,Andorra,1993.0,<NA>
Angola_2010,./data/Angola_2010.txt,Angola,2010,Angola,2010.0,<NA>


In [8]:
# Keep the pattern inventory visible in LIB, as in the M04 example.
LIB["heading_regex"] = "; ".join([pat for _name, pat in heading_pat_list])
LIB.head()


,source_file_path,country,file_year,title,original_year,revision_year,heading_regex
constitution_id,,,,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan,2004,Afghanistan,2004.0,<NA>,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...
Albania_2008,./data/Albania_2008.txt,Albania,2008,Albania,1998.0,2008,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...
Algeria_2008,./data/Algeria_2008.txt,Algeria,2008,Algeria,1963.0,2008,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...
Andorra_1993,./data/Andorra_1993.txt,Andorra,1993,Andorra,1993.0,<NA>,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...
Angola_2010,./data/Angola_2010.txt,Angola,2010,Angola,2010.0,<NA>,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...


## Parser Helpers

The M04 example delegates parsing to `TextParser`. Here we use constitution-specific helper functions because this corpus has legal headings rather than Gutenberg chapter markup.


In [9]:
PREAMBLE_RE = re.compile(r"^Preamble$", re.IGNORECASE)
TOP_HEADING_RE = re.compile(r"^(?P<kind>PART|TITLE|SCHEDULE)\b(?P<rest>.*)$", re.IGNORECASE)
CHAPTER_RE = re.compile(r"^(?P<kind>CHAPTER|Chapter)\b(?P<rest>.*)$")
ARTICLE_RE = re.compile(r"^(?P<kind>ARTICLE|Article)\s+(?P<num>[A-Za-z0-9IVXLCDMivxlcdm]+)\b(?P<rest>.*)$")
SECTION_RE = re.compile(r"^(?P<kind>SECTION|Section)\s+(?P<num>[A-Za-z0-9IVXLCDMivxlcdm]+)\b(?P<rest>.*)$")
NUMBERED_UNIT_RE = re.compile(r"^(?P<num>\d+[A-Z]?)\.\s+(?P<title>\S.*)$")
SPECIAL_UNIT_RE = re.compile(r"^(?P<kind>Explanation|Transitional Provision|Final Provision|Additional Provision)\b(?P<rest>.*)$", re.IGNORECASE)
ROMAN_ONLY_RE = re.compile(r"^[IVXLCDM]+$", re.IGNORECASE)
SENT_SPLIT_RE = re.compile(r"(?<=[.!?;:])\s+(?=[A-Z0-9\"'])")
TOKENIZER = RegexpTokenizer(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:\.[0-9]+)?")


def normalize_heading_text(line):
    return re.sub(r"\s+", " ", line.strip())


def split_heading_label(match):
    kind = match.group("kind").lower()
    label = normalize_heading_text(match.group(0))
    rest = normalize_heading_text(match.groupdict().get("rest") or "")
    heading = rest.lstrip(".:- ") or pd.NA
    return kind, label, heading


def is_blank_or_clipped(line):
    stripped = line.strip()
    return stripped == "" or any(re.search(pat, stripped) for pat in clip_pats)


def split_sentences(para_str):
    sentences = [s.strip() for s in SENT_SPLIT_RE.split(para_str.strip())]
    return [s for s in sentences if s]


def clean_term(token):
    term = re.sub(r"[\W_]+", "", token).lower()
    return term or pd.NA


In [10]:
def parse_units(source_file_path):
    raw_lines = Path(source_file_path).read_text(encoding="utf-8", errors="ignore").splitlines()
    constitution_id, country, file_year = parse_filename(source_file_path)
    title, original_year, revision_year = parse_title_line(raw_lines[0] if raw_lines else Path(source_file_path).stem)

    part_num = 0
    section_num = 0
    unit_num = 0
    part_type = pd.NA
    part_label = pd.NA
    section_type = pd.NA
    section_label = pd.NA

    units = []
    current = None

    def close_current():
        nonlocal current
        if current is not None:
            current["unit_str"] = "\n".join(current.pop("lines")).strip()
            if current["unit_str"]:
                units.append(current)
            current = None

    def open_unit(unit_type, unit_label, unit_heading=pd.NA):
        nonlocal current, unit_num
        close_current()
        unit_num += 1
        current = {
            "constitution_id": constitution_id,
            "part_num": part_num,
            "section_num": section_num,
            "unit_num": unit_num,
            "country": country,
            "file_year": file_year,
            "original_year": original_year,
            "revision_year": revision_year,
            "title": title,
            "source_file_path": source_file_path,
            "part_type": part_type,
            "part_label": part_label,
            "section_type": section_type,
            "section_label": section_label,
            "unit_type": unit_type,
            "unit_label": unit_label,
            "unit_heading": unit_heading,
            "lines": [],
        }

    for raw_line in raw_lines[1:]:
        line = raw_line.strip()
        if is_blank_or_clipped(line):
            if current is not None:
                current["lines"].append("")
            continue

        if PREAMBLE_RE.match(line):
            open_unit("preamble", "Preamble", pd.NA)
            continue

        top_match = TOP_HEADING_RE.match(line)
        if top_match:
            close_current()
            part_num += 1
            section_num = 0
            kind, label, _heading = split_heading_label(top_match)
            part_type = kind
            part_label = label
            section_type = pd.NA
            section_label = pd.NA
            continue

        chapter_match = CHAPTER_RE.match(line)
        if chapter_match:
            close_current()
            kind, label, _heading = split_heading_label(chapter_match)
            if part_num > 0 and str(part_type).lower() in {"part", "title", "schedule"}:
                section_num += 1
                section_type = kind
                section_label = label
            else:
                part_num += 1
                section_num = 0
                part_type = kind
                part_label = label
                section_type = pd.NA
                section_label = pd.NA
            continue

        article_match = ARTICLE_RE.match(line)
        if article_match:
            kind, label, heading = split_heading_label(article_match)
            article_num = article_match.group("num")
            if ROMAN_ONLY_RE.match(article_num) and pd.isna(heading):
                close_current()
                part_num += 1
                section_num = 0
                part_type = kind
                part_label = label
                section_type = pd.NA
                section_label = pd.NA
            else:
                open_unit(kind, label, heading)
                if not pd.isna(heading):
                    current["lines"].append(str(heading))
            continue

        section_match = SECTION_RE.match(line)
        if section_match:
            kind, label, heading = split_heading_label(section_match)
            open_unit(kind, label, heading)
            if not pd.isna(heading):
                current["lines"].append(str(heading))
            continue

        special_match = SPECIAL_UNIT_RE.match(line)
        if special_match:
            kind, label, heading = split_heading_label(special_match)
            open_unit(kind, label, heading)
            if not pd.isna(heading):
                current["lines"].append(str(heading))
            continue

        numbered_match = NUMBERED_UNIT_RE.match(line)
        if numbered_match:
            heading = normalize_heading_text(numbered_match.group("title"))
            open_unit("numbered", numbered_match.group("num"), heading)
            current["lines"].append(heading)
            continue

        if current is None:
            open_unit("text", "Text", pd.NA)
        current["lines"].append(raw_line.rstrip())

    close_current()
    return pd.DataFrame(units)


## Tokenize Corpus

We tokenize each constitution and add each token table to a list to be concatenated into a single `CORPUS`, following the M04 collection-tokenization pattern.


In [11]:
def tokenize_constitution(source_file_path):
    UNITS = parse_units(source_file_path)
    tokens = []
    metadata_cols = [
        "constitution_id", "country", "file_year", "original_year", "revision_year", "title", "source_file_path",
        "part_num", "part_type", "part_label", "section_num", "section_type", "section_label",
        "unit_num", "unit_type", "unit_label", "unit_heading",
    ]

    for unit in UNITS.to_dict("records"):
        base = {col: unit[col] for col in metadata_cols}
        paras = [p.strip() for p in re.split(r"\n\s*\n+", unit["unit_str"]) if p.strip()]
        for para_num, para_str in enumerate(paras, start=1):
            sentence_tokens = []
            for sent_str in split_sentences(para_str):
                sent_tokens = TOKENIZER.tokenize(sent_str)
                if sent_tokens:
                    sentence_tokens.append(sent_tokens)

            if not sentence_tokens:
                continue

            tagged_sentences = nltk.pos_tag_sents(sentence_tokens, lang="eng")
            for sent_num, tagged_sentence in enumerate(tagged_sentences, start=1):
                for token_num, (token_str, pos) in enumerate(tagged_sentence, start=1):
                    tokens.append({
                        **base,
                        "para_num": para_num,
                        "sent_num": sent_num,
                        "token_num": token_num,
                        "token_str": token_str,
                        "term_str": clean_term(token_str),
                        "pos": pos,
                    })

    TOKENS = pd.DataFrame(tokens)
    TOKENS = TOKENS.dropna(subset=["term_str"])
    TOKENS = TOKENS.set_index(OHCO).sort_index()
    return TOKENS, UNITS


def tokenize_collection(LIB):
    corpora = []
    unit_tables = []

    for constitution_id in LIB.index:
        print("Tokenizing", constitution_id, LIB.loc[constitution_id].title)
        source_file_path = LIB.loc[constitution_id].source_file_path
        TOKENS, UNITS = tokenize_constitution(source_file_path)
        corpora.append(TOKENS)
        unit_tables.append(UNITS)

    CORPUS = pd.concat(corpora).sort_index()
    UNITS = pd.concat(unit_tables).sort_values(["constitution_id", "part_num", "section_num", "unit_num"])

    print("Done")
    return CORPUS, UNITS


In [12]:
CORPUS, UNITS = tokenize_collection(LIB)
CORPUS.head()


Tokenizing Afghanistan_2004 Afghanistan
Tokenizing Albania_2008 Albania
Tokenizing Algeria_2008 Algeria
Tokenizing Andorra_1993 Andorra
Tokenizing Angola_2010 Angola
Tokenizing Antigua_and_Barbuda_1981 Antigua and Barbuda
Tokenizing Argentina_1994 Argentina 1853 (reinst. 1983, rev. 1994)
Tokenizing Armenia_2005 Armenia
Tokenizing Australia_1985 Australia
Tokenizing Austria_2009 Austria 1920 (reinst. 1945, rev. 2009)
Tokenizing Azerbaijan_2009 Azerbaijan
Tokenizing Bahamas_2002 Bahamas
Tokenizing Bahrain_2002 Bahrain
Tokenizing Bangladesh_2011 Bangladesh 1972 (reinst. 1986, rev. 2011)
Tokenizing Barbados_2007 Barbados
Tokenizing Belarus_2004 Belarus
Tokenizing Belgium_2012 Belgium
Tokenizing Belize_2001 Belize
Tokenizing Benin_1990 Benin
Tokenizing Bhutan_2008 Bhutan
Tokenizing Bolivia_2009 Bolivia
Tokenizing Bosnia_Herzegovina_2009 Bosnia-Herzegovina
Tokenizing Botswana_2002 Botswana
Tokenizing Brazil_2014 Brazil
Tokenizing Brunei_1984 Brunei
Tokenizing Bulgaria_2007 Bulgaria
Tokenizin

country  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                            file_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num              
Afghanistan_2004 0        0           1        1        1        1               2004   
                                                                 2               2004   
                                                                 3               2004   
                                                                 4               2004   
                                                                 5               2004   

                                                                           original_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                 
Afghanistan_2004 0        0           1        1        1        1                  2004   
                                                                 2                  2004   
                                                                 3                  2004   
                                                                 4                  2004   
                                                                 5                  2004   

                                                                           revision_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                 
Afghanistan_2004 0        0           1        1        1        1                  None   
                                                                 2                  None   
                                                                 3                  None   
                                                                 4                  None   
                                                                 5                  None   

                                                                                  title  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                                       source_file_path  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                                
Afghanistan_2004 0        0           1        1        1        1          ./data/Afghanistan_2004.txt   
                                                                 2          ./data/Afghanistan_2004.txt   
                                                                 3          ./data/Afghanistan_2004.txt   
                                                                 4          ./data/Afghanistan_2004.txt   
                                                                 5          ./data/Afghanistan_2004.txt   

                                                                           part_type  \
constitution_id  part_num section_num unit_num para_

In [13]:
CORPUS.head(20)

country  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   
                                                                 6          Afghanistan   
                                                                 7          Afghanistan   
                                                                 8          Afghanistan   
                                                                 9          Afghanistan   
                                                                 10         Afghanistan   
                                                                 11         Afghanistan   
                                               2        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   
                                                                 6          Afghanistan   
                                                                 7          Afghanistan   
                                                                 8          Afghanistan   
                                                                 9          Afghanistan   

                                                                            file_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num              
Afghanistan_2004 0        0           1        1        1        1               2004   
                                                                 2               2004   
                                                                 3               2004   
                                                                 4               2004   
                                                                 5               2004   
                                                                 6               2004   
                                                                 7               2004   
                                                                 8               2004   
                                                                 9               2004   
                                                                 10              2004   
                                                                 11              2004   
                                               2        1        1               2004   
                                                                 2               2004   
                                                                 3               2004   
                                                                 4               2004   
                                                                 5               2004   
                                                                 6               2004   
                                                                 7               2004   
                                                                 8               2004   
                                                                 9               2004   

                                                                           original_year  \
constitution_id  part_num 

In [14]:
UNITS.head(20)


,constitution_id,part_num,section_num,unit_num,country,file_year,original_year,revision_year,title,source_file_path,part_type,part_label,section_type,section_label,unit_type,unit_label,unit_heading,unit_str
0,Afghanistan_2004,0,0,1,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,NaN,NaN,<NA>,<NA>,preamble,Preamble,<NA>,"In the name of Allah, the Most Beneficent, the Most Merciful\n\n\nPraise be to Allah, the Cherisher and Sustainer of..."
1,Afghanistan_2004,1,0,2,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 1,<NA>,"Afghanistan shall be an Islamic Republic, independent, unitary and indivisible state."
2,Afghanistan_2004,1,0,3,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 2,<NA>,The sacred religion of Islam is the religion of the Islamic Republic of Afghanistan. Followers of other faiths shall...
3,Afghanistan_2004,1,0,4,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 3,<NA>,No law shall contravene the tenets and provisions of the holy religion of Islam in Afghanistan.
4,Afghanistan_2004,1,0,5,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 4,<NA>,"National sovereignty in Afghanistan shall belong to the nation, manifested directly and through its elected represen..."
5,Afghanistan_2004,1,0,6,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 5,<NA>,"Implementing the provisions of this constitution and other laws, defending independence, national sovereignty, terri..."
6,Afghanistan_2004,1,0,7,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 6,<NA>,"The state shall be obligated to create a prosperous and progressive society based on social justice, preservation of..."
7,Afghanistan_2004,1,0,8,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 7,<NA>,"The state shall observe the United Nations Charter, interstate agreements, as well as international treaties to whic..."
8,Afghanistan_2004,1,0,9,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 8,<NA>,"The state shall regulate the foreign policy of the country on the basis of preserving the independence, national int..."
9,Afghanistan_2004,1,0,10,Afghanistan,2004,2004,<NA>,Afghanistan,./data/Afghanistan_2004.txt,chapter,Chapter I. State,<NA>,<NA>,article,Article 9,<NA>,"Mines and other subterranean resources as well as historical relics shall be the property of the state. Protection, ..."


## Extract some features for `LIB`

In [15]:
LIB["constitution_len"] = CORPUS.groupby("constitution_id").term_str.count()
LIB["n_parts"] = CORPUS.reset_index()[["constitution_id", "part_num"]].drop_duplicates().groupby("constitution_id").part_num.count()
LIB["n_sections"] = CORPUS.reset_index()[["constitution_id", "part_num", "section_num"]].drop_duplicates().groupby("constitution_id").section_num.count()
LIB["n_units"] = CORPUS.reset_index()[["constitution_id", "unit_num"]].drop_duplicates().groupby("constitution_id").unit_num.count()
LIB["n_chars"] = LIB.source_file_path.apply(lambda p: len(Path(p).read_text(encoding="utf-8", errors="ignore")))
LIB.sort_values("constitution_len").head()


,source_file_path,country,file_year,title,original_year,revision_year,heading_regex,constitution_len,n_parts,n_sections,n_units,n_chars
constitution_id,,,,,,,,,,,,
Libya_2011,./data/Libya_2011.txt,Libya,2011,Libya,2011.0,<NA>,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,2894,6,6,6,18926
Iceland_1999,./data/Iceland_1999.txt,Iceland,1999,Iceland,1944.0,1999,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,3947,1,1,80,25334
Monaco_2002,./data/Monaco_2002.txt,Monaco,2002,Monaco,1962.0,2002,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,4359,12,12,12,28323
Latvia_2007,./data/Latvia_2007.txt,Latvia,2007,"Latvia 1922 (reinst. 1991, rev. 2007)",NaN,NaN,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,4523,9,9,117,29797
Japan_1946,./data/Japan_1946.txt,Japan,1946,Japan,1946.0,<NA>,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,4735,12,12,104,31269


In [16]:
LIB.query("country == 'United States of America'")

,source_file_path,country,file_year,title,original_year,revision_year,heading_regex,constitution_len,n_parts,n_sections,n_units,n_chars
constitution_id,,,,,,,,,,,,
United_States_of_America_1992,./data/United_States_of_America_1992.txt,United States of America,1992,United States of America,1789.0,1992,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,7523,8,8,58,45800


In [17]:
LIB.describe(include="all")


,source_file_path,country,file_year,title,original_year,revision_year,heading_regex,constitution_len,n_parts,n_sections,n_units,n_chars
count,192,192,192.000000,192,182.000000,127.0,192,192.000000,192.000000,192.000000,192.000000,192.000000
unique,192,192,NaN,192,NaN,30.0,1,NaN,NaN,NaN,NaN,NaN
top,./data/Afghanistan_2004.txt,Afghanistan,NaN,Afghanistan,NaN,2012.0,^Preamble$; ^(?:PART|Part)\b; ^(?:TITLE|Title)\b; ^(?:CHAPTER|Chapter)\b; ^(?:SCHEDULE|Schedule)\b; ^(?:ARTICLE|Arti...,NaN,NaN,NaN,NaN,NaN
freq,1,1,NaN,1,NaN,12.0,192,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,2002.713542,NaN,1980.390110,NaN,NaN,21709.276042,16.661458,22.692708,390.333333,138099.458333
std,NaN,NaN,10.163904,NaN,35.527229,NaN,NaN,15913.882719,21.622292,25.267249,318.754371,98051.371967
min,NaN,NaN,1946.000000,NaN,1789.000000,NaN,NaN,2894.000000,1.000000,1.000000,6.000000,18926.000000
25%,NaN,NaN,1998.750000,NaN,1975.250000,NaN,NaN,10279.000000,8.000000,12.000000,161.750000,66752.000000
50%,NaN,NaN,2005.000000,NaN,1991.000000,NaN,NaN,15639.500000,13.000000,17.000000,274.500000,101157.000000
75%,NaN,NaN,2010.000000,NaN,1999.000000,NaN,NaN,31608.500000,18.000000,27.000000,557.750000,191759.250000


# Extract VOCAB

Extract a vocabulary from the `CORPUS` as a whole.


In [18]:
empty_term_mask = CORPUS.term_str.eq("")
empty_term_mask.sum()


np.int64(0)

In [19]:
CORPUS.loc[empty_term_mask, "token_str"].value_counts().head(20)


Series([], Name: count, dtype: int64)

In [20]:
CORPUS = CORPUS.loc[~empty_term_mask].copy()


In [21]:
# Use the same simple POS grouping style as M04, then add a readable group too.
CORPUS["pos_group"] = CORPUS.pos.str[:2]

pos_group_names = {
    "NN": "NOUN", "VB": "VERB", "JJ": "ADJ", "RB": "ADV",
    "PR": "PRON", "WP": "PRON", "DT": "DET", "PD": "DET", "WD": "DET",
    "IN": "ADP", "TO": "ADP", "CC": "CONJ", "CD": "NUM",
}
CORPUS["pos_group_name"] = CORPUS.pos_group.map(pos_group_names).fillna("OTHER")
CORPUS.head()


country  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                            file_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num              
Afghanistan_2004 0        0           1        1        1        1               2004   
                                                                 2               2004   
                                                                 3               2004   
                                                                 4               2004   
                                                                 5               2004   

                                                                           original_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                 
Afghanistan_2004 0        0           1        1        1        1                  2004   
                                                                 2                  2004   
                                                                 3                  2004   
                                                                 4                  2004   
                                                                 5                  2004   

                                                                           revision_year  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                 
Afghanistan_2004 0        0           1        1        1        1                  None   
                                                                 2                  None   
                                                                 3                  None   
                                                                 4                  None   
                                                                 5                  None   

                                                                                  title  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                
Afghanistan_2004 0        0           1        1        1        1          Afghanistan   
                                                                 2          Afghanistan   
                                                                 3          Afghanistan   
                                                                 4          Afghanistan   
                                                                 5          Afghanistan   

                                                                                       source_file_path  \
constitution_id  part_num section_num unit_num para_num sent_num token_num                                
Afghanistan_2004 0        0           1        1        1        1          ./data/Afghanistan_2004.txt   
                                                                 2          ./data/Afghanistan_2004.txt   
                                                                 3          ./data/Afghanistan_2004.txt   
                                                                 4          ./data/Afghanistan_2004.txt   
                                                                 5          ./data/Afghanistan_2004.txt   

                                                                           part_type  \
constitution_id  part_num section_num unit_num para_

In [22]:
VOCAB = CORPUS.term_str.value_counts().to_frame("n").sort_index()
VOCAB.index.name = "term_str"
VOCAB["n_chars"] = VOCAB.index.str.len()
VOCAB["p"] = VOCAB.n / VOCAB.n.sum()
VOCAB["i"] = -np.log2(VOCAB.p)
VOCAB.head()


,n,n_chars,p,i
term_str,,,,
0,25,1,5.997820e-06,17.347130
00,9,2,2.159215e-06,18.821061
000,210,3,5.038169e-05,14.276741
00000,2,5,4.798256e-07,20.990986
001,5,3,1.199564e-06,19.669058


# Annotate VOCAB

In [23]:
CORPUS[["term_str", "pos"]].value_counts().unstack(fill_value=0).astype(bool).sum(1).sort_values(ascending=False).head(20)


/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_52314/982337760.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  CORPUS[["term_str", "pos"]].value_counts().unstack(fill_value=0).astype(bool).sum(1).sort_values(ascending=False).head(20)


term_str
non          16
twenty       15
o            15
e            15
thereof      15
s            14
amongst      14
sixty        14
therein      14
labour       13
ten          13
etc          13
n            13
thirty       13
twelve       12
paragraph    12
organ        12
c            12
d            11
origin       11
dtype: int64

In [24]:
VOCAB["max_pos"] = CORPUS[["term_str", "pos"]].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB["max_pos_group"] = CORPUS[["term_str", "pos_group"]].value_counts().unstack(fill_value=0).idxmax(1)
VOCAB.head()


,n,n_chars,p,i,max_pos,max_pos_group
term_str,,,,,,
0,25,1,5.997820e-06,17.347130,CD,CD
00,9,2,2.159215e-06,18.821061,CD,CD
000,210,3,5.038169e-05,14.276741,CD,CD
00000,2,5,4.798256e-07,20.990986,CD,CD
001,5,3,1.199564e-06,19.669058,CD,CD


In [25]:
VOCAB["n_pos_group"] = CORPUS[["term_str", "pos_group"]].value_counts().unstack().count(1)
VOCAB["cat_pos_group"] = CORPUS[["term_str", "pos_group"]].value_counts().to_frame("n").reset_index()\
    .groupby("term_str").pos_group.apply(lambda x: set(x))
VOCAB["n_pos"] = CORPUS[["term_str", "pos"]].value_counts().unstack().count(1)
VOCAB["cat_pos"] = CORPUS[["term_str", "pos"]].value_counts().to_frame("n").reset_index()\
    .groupby("term_str").pos.apply(lambda x: set(x))
VOCAB.head()


,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos
term_str,,,,,,,,,,
0,25,1,5.997820e-06,17.347130,CD,CD,1,{CD},1,{CD}
00,9,2,2.159215e-06,18.821061,CD,CD,1,{CD},1,{CD}
000,210,3,5.038169e-05,14.276741,CD,CD,1,{CD},1,{CD}
00000,2,5,4.798256e-07,20.990986,CD,CD,1,{CD},1,{CD}
001,5,3,1.199564e-06,19.669058,CD,CD,1,{CD},1,{CD}


In [26]:
sw = pd.Series(1, index=stopwords.words("english"), name="stop")
VOCAB["stop"] = VOCAB.index.map(sw).fillna(0).astype("int")
VOCAB.stop.sum()


np.int64(135)

In [27]:
stemmer = PorterStemmer()
VOCAB["porter_stem"] = VOCAB.apply(lambda x: stemmer.stem(x.name), 1)
VOCAB.head()


,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem
term_str,,,,,,,,,,,,
0,25,1,5.997820e-06,17.347130,CD,CD,1,{CD},1,{CD},0,0
00,9,2,2.159215e-06,18.821061,CD,CD,1,{CD},1,{CD},0,00
000,210,3,5.038169e-05,14.276741,CD,CD,1,{CD},1,{CD},0,000
00000,2,5,4.798256e-07,20.990986,CD,CD,1,{CD},1,{CD},0,00000
001,5,3,1.199564e-06,19.669058,CD,CD,1,{CD},1,{CD},0,001


# Add DFIDF

The final project asks for `dfidf` in `VOCAB`. This follows the M5 aggregate TFIDF notebook without creating `BOW`, `DTCM`, or `TFIDF` tables here.

Document frequency `df` is the number of constitution documents in which a term appears at least once. Following M5, `dfidf = df * idf`. This will produce ties because it depends only on document frequency, not total term frequency. The related `dh` column is document entropy, which M5 uses as a colinear significance measure.


In [28]:
DOC = CORPUS.groupby("constitution_id").term_str.count().to_frame("n_tokens")
DOC["n_types"] = CORPUS.reset_index()[["constitution_id", "term_str"]].drop_duplicates().groupby("constitution_id").term_str.count()
DOC["pkr"] = DOC.n_types / DOC.n_tokens
DOC = DOC.join(LIB[["country", "file_year", "original_year", "revision_year", "title"]])
DOC.sort_values("pkr").head(20)


,n_tokens,n_types,pkr,country,file_year,original_year,revision_year,title
constitution_id,,,,,,,,
India_2012,101401,3725,0.036735,India,2012,1949.0,2012,India
St_Kitts_and_Nevis_1983,45856,2072,0.045185,St Kitts and Nevis,1983,1983.0,<NA>,St. Kitts and Nevis
Malaysia_1996,62859,3049,0.048505,Malaysia,1996,1957.0,1996,Malaysia
St_Lucia_1978,39683,1950,0.049139,St Lucia,1978,1978.0,<NA>,St. Lucia
Fiji_2013,46381,2337,0.050387,Fiji,2013,2013.0,<NA>,Fiji
Jamaica_1994,37714,1939,0.051413,Jamaica,1994,1962.0,1994,Jamaica
Singapore_2010,43355,2231,0.051459,Singapore,2010,1959.0,2010,Singapore
Lesotho_1998,42028,2185,0.051989,Lesotho,1998,1993.0,1998,Lesotho
Sweden_2012,57609,3043,0.052822,Sweden,2012,1974.0,2012,Sweden


In [29]:
# Number of constitution documents
N = DOC.shape[0]

DF = CORPUS.reset_index()[["constitution_id", "term_str"]].drop_duplicates().term_str.value_counts().sort_index()
IDF = np.log2(N / DF)

VOCAB["df"] = DF
VOCAB["idf"] = IDF
VOCAB["dfidf"] = VOCAB.df * VOCAB.idf
VOCAB["dp"] = VOCAB.df / N
VOCAB["di"] = np.log2(1 / VOCAB.dp)
VOCAB["dh"] = VOCAB.dp * VOCAB.di

VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)


,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh
term_str,,,,,,,,,,,,,,,,,,
assent,622,6,0.000149,12.710216,NN,NN,3,"{JJ, VB, NN}",5,"{VBD, VB, JJ, NN, NNP}",0,assent,71,1.435215,101.900292,0.369792,1.435215,0.530731
labor,496,5,0.000119,13.036790,NN,NN,1,{NN},2,"{NN, NNP}",0,labor,71,1.435215,101.900292,0.369792,1.435215,0.530731
rural,397,5,0.000095,13.357991,JJ,JJ,3,"{JJ, NN, VB}",3,"{VB, JJ, NNP}",0,rural,71,1.435215,101.900292,0.369792,1.435215,0.530731
hand,348,4,0.000083,13.548043,NN,NN,1,{NN},2,"{NN, NNP}",0,hand,71,1.435215,101.900292,0.369792,1.435215,0.530731
like,303,4,0.000073,13.747813,JJ,JJ,3,"{JJ, IN, VB}",3,"{JJ, IN, VB}",0,like,71,1.435215,101.900292,0.369792,1.435215,0.530731
organized,299,9,0.000072,13.766985,VBN,VB,3,"{JJ, VB, NN}",4,"{VBN, JJ, VBD, NNP}",0,organ,71,1.435215,101.900292,0.369792,1.435215,0.530731
class,291,5,0.000070,13.806111,NN,NN,1,{NN},2,"{NN, NNP}",0,class,71,1.435215,101.900292,0.369792,1.435215,0.530731
fees,251,4,0.000060,14.019443,NNS,NN,1,{NN},1,{NNS},0,fee,71,1.435215,101.900292,0.369792,1.435215,0.530731
magistrate,233,10,0.000056,14.126800,NN,NN,3,"{JJ, NN, VB}",5,"{NN, JJ, VB, VBP, NNP}",0,magistr,71,1.435215,101.900292,0.369792,1.435215,0.530731


# Inspect

In [30]:
sample_ids = [
    "United_States_of_America_1992",
    "India_2012",
    "France_2008",
    "South_Africa_2012",
]
UNITS.loc[UNITS.constitution_id.isin(sample_ids), [
    "constitution_id", "part_num", "part_label", "section_num", "section_label",
    "unit_num", "unit_type", "unit_label", "unit_heading", "unit_str"
]].head(40)


,constitution_id,part_num,part_label,section_num,section_label,unit_num,unit_type,unit_label,unit_heading,unit_str
0,France_2008,0,NaN,0,<NA>,1,preamble,Preamble,NaN,The French people solemnly proclaim their attachment to the Rights of Man and the principles of national sovereignty...
1,France_2008,0,NaN,0,<NA>,2,article,ARTICLE 1,NaN,"France shall be an indivisible, secular, democratic and social Republic. It shall ensure the equality of all citizen..."
2,France_2008,1,Title I. ON SOVEREIGNTY,0,<NA>,3,article,ARTICLE 2,NaN,"The language of the Republic shall be French.\n\n\nThe national emblem shall be the blue, white and red tricolour fl..."
3,France_2008,1,Title I. ON SOVEREIGNTY,0,<NA>,4,article,ARTICLE 3,NaN,"National sovereignty shall vest in the people, who shall exercise it through their representatives and by means of r..."
4,France_2008,1,Title I. ON SOVEREIGNTY,0,<NA>,5,article,ARTICLE 4,NaN,Political parties and groups shall contribute to the exercise of suffrage. They shall be formed and carry on their a...
5,France_2008,2,Title II. THE PRESIDENT OF THE REPUBLIC,0,<NA>,6,article,ARTICLE 5,NaN,"The President of the Republic shall ensure due respect for the Constitution. He shall ensure, by his arbitration, th..."
6,France_2008,2,Title II. THE PRESIDENT OF THE REPUBLIC,0,<NA>,7,article,ARTICLE 6,NaN,The President of the Republic shall be elected for a terra of five years by direct universal suffrage.\n\n\nNo one m...
7,France_2008,2,Title II. THE PRESIDENT OF THE REPUBLIC,0,<NA>,8,article,ARTICLE 7,NaN,The President of the Republic shall be elected by an absolute majority of votes cast. If such a majority is not obta...
8,France_2008,2,Title II. THE PRESIDENT OF THE REPUBLIC,0,<NA>,9,article,ARTICLE 8,NaN,The President of the Republic shall appoint the Prime Minister. He shall terminate the appointment of the Prime Mini...
9,France_2008,2,Title II. THE PRESIDENT OF THE REPUBLIC,0,<NA>,10,article,ARTICLE 9,NaN,The President of the Republic shall preside over the Council of Ministers.


In [31]:
CORPUS.reset_index().query("constitution_id in @sample_ids").head(40)


,constitution_id,part_num,section_num,unit_num,para_num,sent_num,token_num,country,file_year,original_year,revision_year,title,source_file_path,part_type,part_label,section_type,section_label,unit_type,unit_label,unit_heading,token_str,term_str,pos,pos_group,pos_group_name
1184291,France_2008,0,0,1,1,1,1,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,The,the,DT,DT,DET
1184292,France_2008,0,0,1,1,1,2,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,French,french,JJ,JJ,ADJ
1184293,France_2008,0,0,1,1,1,3,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,people,people,NNS,NN,NOUN
1184294,France_2008,0,0,1,1,1,4,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,solemnly,solemnly,RB,RB,ADV
1184295,France_2008,0,0,1,1,1,5,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,proclaim,proclaim,VBP,VB,VERB
1184296,France_2008,0,0,1,1,1,6,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,their,their,PRP$,PR,PRON
1184297,France_2008,0,0,1,1,1,7,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,attachment,attachment,NN,NN,NOUN
1184298,France_2008,0,0,1,1,1,8,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,to,to,TO,TO,ADP
1184299,France_2008,0,0,1,1,1,9,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,the,the,DT,DT,DET
1184300,France_2008,0,0,1,1,1,10,France,2008,1958,2008,France,./data/France_2008.txt,NaN,NaN,None,None,preamble,Preamble,NaN,Rights,rights,NNPS,NN,NOUN


In [32]:
VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)


,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh
term_str,,,,,,,,,,,,,,,,,,
assent,622,6,0.000149,12.710216,NN,NN,3,"{JJ, VB, NN}",5,"{VBD, VB, JJ, NN, NNP}",0,assent,71,1.435215,101.900292,0.369792,1.435215,0.530731
labor,496,5,0.000119,13.036790,NN,NN,1,{NN},2,"{NN, NNP}",0,labor,71,1.435215,101.900292,0.369792,1.435215,0.530731
rural,397,5,0.000095,13.357991,JJ,JJ,3,"{JJ, NN, VB}",3,"{VB, JJ, NNP}",0,rural,71,1.435215,101.900292,0.369792,1.435215,0.530731
hand,348,4,0.000083,13.548043,NN,NN,1,{NN},2,"{NN, NNP}",0,hand,71,1.435215,101.900292,0.369792,1.435215,0.530731
like,303,4,0.000073,13.747813,JJ,JJ,3,"{JJ, IN, VB}",3,"{JJ, IN, VB}",0,like,71,1.435215,101.900292,0.369792,1.435215,0.530731
organized,299,9,0.000072,13.766985,VBN,VB,3,"{JJ, VB, NN}",4,"{VBN, JJ, VBD, NNP}",0,organ,71,1.435215,101.900292,0.369792,1.435215,0.530731
class,291,5,0.000070,13.806111,NN,NN,1,{NN},2,"{NN, NNP}",0,class,71,1.435215,101.900292,0.369792,1.435215,0.530731
fees,251,4,0.000060,14.019443,NNS,NN,1,{NN},1,{NNS},0,fee,71,1.435215,101.900292,0.369792,1.435215,0.530731
magistrate,233,10,0.000056,14.126800,NN,NN,3,"{JJ, NN, VB}",5,"{NN, JJ, VB, VBP, NNP}",0,magistr,71,1.435215,101.900292,0.369792,1.435215,0.530731


# Save

The export lines are intentionally commented out so the tables can be inspected before saving.


In [33]:
save_path = f"{output_dir}/{data_prefix}"

Path(output_dir).mkdir(exist_ok=True)
#LIB.to_csv(f"{save_path}-LIB.csv", sep=CSV_DELIM)
#CORPUS.to_csv(f"{save_path}-CORPUS.csv", sep=CSV_DELIM)
#VOCAB.to_csv(f"{save_path}-VOCAB.csv", sep=CSV_DELIM)
#DOC.to_csv(f"{save_path}-DOC.csv", sep=CSV_DELIM)
